<a href="https://colab.research.google.com/github/Pauloade123/ML-intern/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Lane 4: This a regression/scoring.

Why: It allows you to output a dynamically ranked queue. Editors can sort pages from highest opportunity score to lowest, taking action on the top 10 or 20 pages each week regardless of arbitrary cutoff lines

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

# 1. Load starter dataset
url = "https://raw.githubusercontent.com/Pauloade123/ML-intern/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# 2. Compute expected CTR benchmark per position tier
df['position_benchmark'] = df.groupby('position_tier')['ctr'].transform('mean')

# 3. Construct target feature: Opportunity Score (Underperformance Gap x Exposure Scale)
df['opportunity_score'] = (df['position_benchmark'] - df['ctr']) * df['impressions_90d']

# 4. View unit of analysis with raw metrics and derived target feature
df[['content_id', 'position_tier', 'avg_position', 'ctr', 'position_benchmark', 'impressions_90d', 'opportunity_score']].head()

,content_id,position_tier,avg_position,ctr,position_benchmark,impressions_90d,opportunity_score
0,content_304f48230142,striking,10.6,0.76,0.323239,3803,-1661.000863
1,content_a1fb4e703a9e,page_3_5,20.3,0.05,0.222484,15320,2642.456725
2,content_9aa793d4d895,page_3_5,36.5,0.09,0.222484,12581,1666.782719
3,content_331d6c4de07b,page_1,6.2,0.49,0.652467,11751,1909.144606
4,content_d99b7a2d90ca,page_3_5,44.0,0.13,0.222484,19140,1770.146065


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary Evaluation Metrics: MAE (Mean Absolute Error) and NDCG@K (Normalized Discounted Cumulative Gain)

Why:

MAE (Mean Absolute Error): Measures how close predicted opportunity scores are to actual scores in plain, interpretable terms (average error in opportunity clicks per page). Lower MAE means more accurate magnitude predictions.

NDCG@10 / NDCG@20: Evaluates the quality of the ranked queue. Editorial teams usually act on the top 10 or 20 pages per batch; NDCG measures whether the model successfully places the true highest-opportunity pages at the top of that queue

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
from sklearn.metrics import mean_absolute_error, ndcg_score

# Example check: Comparing dummy baseline predictions against true target scores
y_true = np.array([[500, 1200, 50, 300, 2200]])
y_pred = np.array([[450, 1100, 80, 250, 2000]])

mae = mean_absolute_error(y_true[0], y_pred[0])
ndcg = ndcg_score(y_true, y_pred, k=3)

print(f"Sample MAE: {mae:.2f} clicks off on average")
print(f"Sample NDCG@3: {ndcg:.4f} ranking effectiveness")


Sample MAE: 86.00 clicks off on average
Sample NDCG@3: 1.0000 ranking effectiveness


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of Analysis: 1 row = 1 unique content item (web page URL).

Explanation: Every observation in the dataset represents a single web page over a trailing 90-day window, identified by its primary key content_id (or url). Predictions and rankings are calculated on a per-page basis so the editorial team can directly audit and refresh individual URLs

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

unit_of_analysis_df = df[['content_id', 'position_tier', 'impressions_90d', 'ctr', 'opportunity_score']].head()
unit_of_analysis_df

,content_id,position_tier,impressions_90d,ctr,opportunity_score
0,content_304f48230142,striking,3803,0.76,-1661.000863
1,content_a1fb4e703a9e,page_3_5,15320,0.05,2642.456725
2,content_9aa793d4d895,page_3_5,12581,0.09,1666.782719
3,content_331d6c4de07b,page_1,11751,0.49,1909.144606
4,content_d99b7a2d90ca,page_3_5,19140,0.13,1770.146065


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Non-Linear Interactions: Fixed heuristic rules (e.g., IF ctr < 1% AND impressions > 5000) fall short because expected performance varies dynamically across search positions, intent types, and content categories. A single static threshold creates false positives for low-position pages and misses major optimization opportunities on high-position pages.

Continuous Trade-Offs: Rule-based logic relies on arbitrary cutoff lines that create sharp edge cases. Machine learning models continuously weigh multiple overlapping features—such as search volume, position decay, intent tier, and engagement rates—to output a smooth, fine-grained priority score across the entire portfolio.

Scalable Ranking Queue: Instead of producing a binary, unranked bucket of "failed" pages, an ML scoring model dynamically ranks the entire catalog, enabling editorial teams to tackle the top 10 or 20 highest-impact pages every week


In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.